# SIEWS+ — Labeled Safety (Belt/Helmet/Vest) Training
## YOLO26n Transfer Learning — Stage 2 PPE

**Dataset:** labeled_safety.v1i.yolo26 — 4740 images

| ID | Class | Instances |
|---|---|---|
| 0 | `.` | 0 (tidak dipakai) |
| 1 | **belt** (safety belt) | 134 |
| 2 | **helmet** | 765 |
| 3 | **vest** | 544 |

**Output:** `best_stage2_labeled_safety.pt`

**Kaggle:** Upload ZIP → Add Data → GPU T4 → Run All → Download `.pt`

## 1. Setup

In [ ]:
import os, sys, subprocess, zipfile, shutil, random, yaml, traceback
from pathlib import Path
from collections import Counter

IS_KAGGLE = os.path.exists('/kaggle/input')
print(f'Kaggle: {IS_KAGGLE}')

subprocess.run([sys.executable,'-m','pip','install','-U','ultralytics','-q'], check=True)
import ultralytics, torch
print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'

## 2. Path Config

In [ ]:
ZIP_NAME = 'labeled_safety.v1i.yolo26.zip'
RUN_NAME = 'labeled_safety'
OUT_NAME = 'best_stage2_labeled_safety.pt'

if IS_KAGGLE:
    DATASET_ZIP = None
    OUTPUT_DIR  = Path('/kaggle/working')
    EXTRACT_DIR = None
else:
    DATASET_ZIP = Path('F:/migas-siews/dataset') / ZIP_NAME
    OUTPUT_DIR  = Path('F:/migas-siews/training/runs_labeled_safety')
    EXTRACT_DIR = OUTPUT_DIR / 'dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / RUN_NAME
if DATASET_ZIP:
    print(f'ZIP    : {DATASET_ZIP}')
    print(f'Exists : {DATASET_ZIP.exists()}')
else:
    print('Kaggle mode')

## 3. Ekstrak Dataset

In [ ]:
if IS_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    found = None
    for candidate in kaggle_input.rglob('train'):
        if (candidate / 'images').exists():
            found = candidate.parent
            break
    if found is None:
        subdirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
        print('Isi /kaggle/input/:', [d.name for d in subdirs])
        found = subdirs[0] if subdirs else None
    if found is None:
        raise FileNotFoundError('Dataset tidak ditemukan! Pastikan sudah Add Data di Kaggle.')
    EXTRACT_DIR = found
    print(f'Dataset ditemukan: {EXTRACT_DIR}')

elif DATASET_ZIP and DATASET_ZIP.exists():
    if not EXTRACT_DIR.exists():
        print('Extracting...')
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        print('Done!')
    else:
        print('Already extracted')
else:
    raise FileNotFoundError(f'ZIP tidak ada: {DATASET_ZIP}')

print()
for split in ['train', 'valid', 'val', 'test']:
    p = EXTRACT_DIR / split / 'images'
    if p.exists():
        ni = len(list(p.glob('*.*')))
        nl_dir = EXTRACT_DIR / split / 'labels'
        nl = len(list(nl_dir.glob('*.txt'))) if nl_dir.exists() else 0
        print(f'[{split:6s}] images={ni:5d}  labels={nl:5d}')

## 4. Verifikasi Class Distribution

In [ ]:
# Class names sudah diverifikasi dari data.yaml dataset
# 0: '.' (kosong - tidak ada instance)
# 1: belt  (safety belt / harness)
# 2: helmet
# 3: vest
CLASS_NAMES = {
    0: 'unknown',  # class '.' tidak ada instance
    1: 'belt',
    2: 'helmet',
    3: 'vest',
}
NC = 4

# Hitung distribusi dari label files
counts = Counter()
lbl_dir = EXTRACT_DIR / 'train' / 'labels'
for lf in lbl_dir.glob('*.txt'):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except: pass

print('Class distribution:')
total = sum(counts.values())
for cid in sorted(CLASS_NAMES):
    cnt = counts.get(cid, 0)
    pct = cnt/total*100 if total else 0
    bar = '#' * (cnt//30)
    print(f'  [{cid}] {CLASS_NAMES[cid]:10s}: {cnt:5d} instances ({pct:4.1f}%)  {bar}')

print(f'\nTotal instances : {total}')
print(f'Max class ID    : {max(counts) if counts else -1} | NC={NC}')

## 5. STEP 1 — Visualisasi Class ID (Angka Dulu!)

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

COLOR_PALETTE = [
    (150, 150, 150),  # ID 0 — abu
    (  0, 200, 255),  # ID 1 — cyan
    (  0, 255,   0),  # ID 2 — green
    (255, 140,   0),  # ID 3 — orange
    (180,   0, 255),  # ID 4 — purple
    (255,   0,   0),  # ID 5 — red
]

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
all_imgs = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
if not all_imgs:
    raise FileNotFoundError(f'Tidak ada gambar di {train_img_dir}')

samples = random.sample(all_imgs, min(9, len(all_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, ip in zip(axes.flat, samples):
    img = cv2.imread(str(ip))
    if img is None: ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    ids_in_img = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 3)
            # Tampilkan ID ANGKA saja — belum ada nama
            cv2.putText(img, f'ID:{cid}', (x1, max(y1-5,18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
            ids_in_img.append(cid)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'IDs ditemukan: {sorted(set(ids_in_img))}', fontsize=10, fontweight='bold')

plt.suptitle('STEP 1 — Lihat class ID, cocokan dengan CLASS_NAMES di cell berikutnya!',
             fontsize=12, fontweight='bold', color='red')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'verify_id.png'), dpi=100)
plt.show()

print('LEGEND warna:')
print('  abu     = ID 0')
print('  cyan    = ID 1')
print('  green   = ID 2')
print('  orange  = ID 3')
print()
print('Pastikan CLASS_NAMES di bawah sudah cocok dengan yang Anda lihat!')

## 5. STEP 2 — Konfirmasi CLASS_NAMES (Isi sesuai STEP 1)

In [ ]:
# ============================================================
# Verifikasi dari data.yaml dataset:
#   0 = '.'       (tidak ada instance di dataset)
#   1 = 'belt'    (safety belt / harness)
#   2 = 'helmet'
#   3 = 'vest'
#
# Cocokkan dengan visualisasi STEP 1 di atas!
# Jika ada yang tidak sesuai, ganti nama di sini.
# ============================================================
CLASS_NAMES = {
    0: 'unknown',
    1: 'belt',
    2: 'helmet',
    3: 'vest',
}
NC = 4

# Re-visualisasi dengan nama untuk konfirmasi
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
samples2 = random.sample(all_imgs, min(9, len(all_imgs)))

for ax, ip in zip(axes.flat, samples2):
    img = cv2.imread(str(ip))
    if img is None: ax.axis('off'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = train_lbl_dir / (ip.stem + '.txt')
    names_found = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
            x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
            col = COLOR_PALETTE[cid % len(COLOR_PALETTE)]
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 2)
            nm = CLASS_NAMES.get(cid, f'cls_{cid}')
            cv2.putText(img, nm, (x1, max(y1-5,15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
            names_found.append(nm)
    ax.imshow(img); ax.axis('off')
    ax.set_title(', '.join(set(names_found)) or '-', fontsize=9)

plt.suptitle('STEP 2 — Konfirmasi nama class. Jika benar, lanjut ke Training!',
             fontsize=12, fontweight='bold', color='blue')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/'confirmed_classes.png'), dpi=100)
plt.show()

# Hitung distribusi
counts = Counter()
for lf in train_lbl_dir.glob('*.txt'):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except: pass
total = sum(counts.values())
print('Class distribution:')
for cid in sorted(CLASS_NAMES):
    cnt = counts.get(cid, 0)
    pct = cnt/total*100 if total else 0
    print(f'  [{cid}] {CLASS_NAMES[cid]:10s}: {cnt:5d} instances ({pct:4.1f}%)')
print()
print('Jika nama sudah benar -> lanjut ke Cell berikutnya (data.yaml + Training)')

In [ ]:
val_split  = 'valid' if (EXTRACT_DIR/'valid'/'images').exists() else 'val'
test_split = 'test'  if (EXTRACT_DIR/'test'/'images').exists()  else val_split

DATA_YAML = OUTPUT_DIR / f'{RUN_NAME}.yaml'
yaml.dump({
    'path' : str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : f'{test_split}/images',
    'nc'   : NC,
    'names': CLASS_NAMES,
}, open(DATA_YAML,'w'), default_flow_style=False)
print(DATA_YAML.read_text())

## 7. Training

**Catatan dataset:**
- Total: 4740 gambar — ukuran bagus!
- `belt` hanya 9.3% → imbalanced. Augmentasi tinggi membantu.
- Sudah di-resize 640x640 oleh Roboflow.

In [ ]:
from ultralytics import YOLO

EPOCHS   = 100
BATCH    = 16     # kurangi ke 8 jika OOM
IMGSZ    = 640
PATIENCE = 0      # disable EarlyStopping
FREEZE   = 10     # transfer learning

print(f'Config: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'EarlyStopping: DISABLED')
print(f'Class imbalance note: belt hanya 9.3% — augmentasi tinggi aktif')

TRAIN_OK = False
try:
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, patience=PATIENCE, freeze=FREEZE, pretrained=True,
        project=str(OUTPUT_DIR/'runs'), name=RUN_NAME, exist_ok=True,
        # Augmentasi untuk deteksi manusia + PPE
        augment=True, mosaic=1.0, mixup=0.1, copy_paste=0.15,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        fliplr=0.5, flipud=0.0, degrees=5.0, scale=0.5, erasing=0.3,
        # Class weights otomatis dari ultralytics
        optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005, warmup_epochs=3,
        save_period=10, verbose=True, plots=True,
    )
    print('\nTraining selesai!')
    TRAIN_OK = True
except KeyboardInterrupt:
    print('Dihentikan manual — checkpoint:', RUN_DIR/'weights'/'last.pt')
except Exception as e:
    print(f'Error: {e}'); traceback.print_exc()

## 8. Evaluasi

In [ ]:
best_pt = RUN_DIR/'weights'/'best.pt'
last_pt = RUN_DIR/'weights'/'last.pt'
eval_pt = best_pt if best_pt.exists() else last_pt

if not eval_pt.exists():
    print('Tidak ada checkpoint!')
else:
    try:
        if TRAIN_OK:
            m = results.results_dict
            map50=m.get('metrics/mAP50(B)',0); prec=m.get('metrics/precision(B)',0); rec=m.get('metrics/recall(B)',0)
        else:
            em=YOLO(str(eval_pt)); vr=em.val(data=str(DATA_YAML),device=DEVICE,verbose=False)
            map50=vr.box.map50; prec=vr.box.mp; rec=vr.box.mr
        status = 'OK' if map50>=0.70 else ('Cukup' if map50>=0.50 else 'Rendah')
        print(f'mAP50     : {map50:.4f}  [{status}]')
        print(f'Precision : {prec:.4f}')
        print(f'Recall    : {rec:.4f}')
        if map50 < 0.50:
            print('\nTips: Coba freeze=5 atau freeze=0 untuk fine-tune lebih dalam')
    except Exception as e:
        print(f'Eval error: {e}')

from IPython.display import Image as IPImg, display
for pn in ['results.png','confusion_matrix.png','PR_curve.png']:
    pf = RUN_DIR/pn
    if pf.exists(): print(f'--- {pn} ---'); display(IPImg(str(pf),width=750))

## 9. Inference Test

In [ ]:
if eval_pt.exists():
    bm = YOLO(str(eval_pt))
    tdir = EXTRACT_DIR/'test'/'images'
    if not tdir.exists(): tdir = EXTRACT_DIR/val_split/'images'
    imgs = list(tdir.glob('*.jpg')) + list(tdir.glob('*.png'))
    samples3 = random.sample(imgs, min(9, len(imgs)))
    fig,axes=plt.subplots(3,3,figsize=(15,12))
    for ax,ip in zip(axes.flat,samples3):
        try:
            pred=bm.predict(str(ip),conf=0.25,verbose=False)[0]
            ax.imshow(cv2.cvtColor(pred.plot(),cv2.COLOR_BGR2RGB))
            bxs=pred.boxes
            title=', '.join([f"{pred.names[int(b.cls[0])]}:{float(b.conf[0]):.0%}" for b in bxs]) if bxs and len(bxs) else 'no det'
            ax.set_title(title[:45],fontsize=8)
        except Exception as e: ax.set_title(str(e)[:30])
        ax.axis('off')
    plt.suptitle('Inference Test — belt / helmet / vest',fontsize=13)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR/'inference.png'),dpi=100)
    plt.show()

## 10. Auto-Save Model

In [ ]:
dest = Path('/kaggle/working') if IS_KAGGLE else Path('F:/migas-siews/backend/models')
dest.mkdir(parents=True, exist_ok=True)

for src, name in [(best_pt, OUT_NAME), (last_pt, OUT_NAME.replace('best_','last_'))]:
    if src.exists():
        dst = dest / name
        shutil.copy2(src, dst)
        print(f'Saved: {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'Tidak ada: {src}')

if IS_KAGGLE:
    print(f'\nLangkah selanjutnya:')
    print(f'  1. Download {OUT_NAME} dari tab Output')
    print(f'  2. Salin ke: f:/migas-siews/backend/models/')
    print(f'  3. Update detector.py S2: ganti ke {OUT_NAME}')
    print(f'  4. Update PPE_CLASSES di detector.py: 1=belt, 2=helmet, 3=vest')
    print(f'  5. Restart backend')

## Catatan Integrasi ke detector.py

Setelah training, update `PPE_CLASSES` di `detector.py`:

```python
PPE_CLASSES: Dict[int, str] = {
    0: 'unknown',
    1: 'belt',      # safety belt / harness
    2: 'helmet',
    3: 'vest',
}
```

Dan update S2 model path:
```python
self.model_s2 = _load_model(
    MODELS_DIR / 'best_stage2_labeled_safety.pt',
    'yolo26n.pt',
    allow_generic_fallback=False,
)
```

**Peringatan class imbalance:**
- `belt` hanya 9.3% → recall untuk belt mungkin rendah
- Jika belt/harness penting untuk kompetisi, cari dataset tambahan